# Customer 360 Accelerator
### Notebook 02 — Create Ontology

---

> **Purpose:** Create a Fabric Ontology with a **Customer** entity type,
> bind it to the `Customer360` Delta table in the Lakehouse, and set up
> proper property types. The Ontology gives the Data Agent semantic
> understanding of the data model.
>
> Each run is idempotent — an existing ontology with the same name is
> deleted and recreated.
>
> **Prerequisite:** Run `00_setup_and_load_data` and `01_create_semantic_model` first.

---
## Configuration

In [ ]:
# ── Paste outputs from previous notebooks, or leave blank to auto-detect ─────
WORKSPACE_ID      = ""
WORKSPACE_NAME    = ""
LAKEHOUSE_ID      = ""
LAKEHOUSE_NAME    = "Customer360Lakehouse"
TABLE_NAME        = "Customer360"
SEMANTIC_MODEL_ID = ""   # Optional: if set, ontology links to the semantic model

ONTOLOGY_NAME     = "Customer360Ontology"

In [ ]:
import sempy.fabric as fabric
import json, base64, uuid, random, time

client = fabric.FabricRestClient()

# Auto-detect workspace
if not WORKSPACE_ID:
    WORKSPACE_ID = fabric.get_notebook_workspace_id()
if not WORKSPACE_NAME:
    WORKSPACE_NAME = fabric.resolve_workspace_name()

# Auto-detect lakehouse
if not LAKEHOUSE_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
    resp.raise_for_status()
    lh = next(
        (l for l in resp.json().get("value", []) if l["displayName"] == LAKEHOUSE_NAME),
        None,
    )
    if not lh:
        raise ValueError(f"Lakehouse '{LAKEHOUSE_NAME}' not found. Run Notebook 00 first.")
    LAKEHOUSE_ID = lh["id"]

# Auto-detect semantic model
if not SEMANTIC_MODEL_ID:
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/items?type=SemanticModel")
    resp.raise_for_status()
    sm = next(
        (i for i in resp.json().get("value", []) if i["displayName"] == LAKEHOUSE_NAME),
        None,
    )
    if sm:
        SEMANTIC_MODEL_ID = sm["id"]

print(f"Workspace      : {WORKSPACE_NAME} ({WORKSPACE_ID})")
print(f"Lakehouse      : {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")
print(f"Semantic Model : {SEMANTIC_MODEL_ID or '(not found — will use Lakehouse binding)'}")

---
## Step 1 — Read Table Schema from Lakehouse

We read the Delta table schema using Spark so we can build the
ontology entity with the correct property types.

In [ ]:
# Spark type → Ontology valueType mapping
SPARK_TO_ONTOLOGY = {
    "string":    "String",
    "boolean":   "Boolean",
    "date":      "DateTime",
    "timestamp": "DateTime",
    "int":       "BigInt",
    "integer":   "BigInt",
    "long":      "BigInt",
    "bigint":    "BigInt",
    "float":     "Double",
    "double":    "Double",
    "decimal":   "Double",
}

# Read schema from the Delta table
full_table = f"{WORKSPACE_NAME}.{LAKEHOUSE_NAME}.{TABLE_NAME}"
schema = spark.table(full_table).schema

columns = []
for field in schema.fields:
    type_str = field.dataType.simpleString()
    ontology_type = SPARK_TO_ONTOLOGY.get(type_str)
    if ontology_type:
        columns.append({"name": field.name, "valueType": ontology_type})
    else:
        print(f"   Skipping {TABLE_NAME}.{field.name} (unsupported type: {type_str})")

print(f"Schema: {len(columns)} compatible columns from {TABLE_NAME}")
for col in columns:
    print(f"   {col['name']:<25} {col['valueType']}")

---
## Step 2 — Build Ontology Definition

We construct the ontology definition payload following Fabric's format:
- **Entity type** with properties mapped from the table columns
- **Data binding** linking the entity to the Lakehouse table
- All payloads are base64-encoded as required by the ontologies API

In [ ]:
random.seed(42)
_used_ids = set()

def generate_id():
    """Generate a unique positive 64-bit integer ID (as a string)."""
    while True:
        id_val = random.randint(10**12, 10**15)
        if id_val not in _used_ids:
            _used_ids.add(id_val)
            return str(id_val)

def to_base64(obj):
    return base64.b64encode(json.dumps(obj).encode("utf-8")).decode("utf-8")


# ── Entity configuration ─────────────────────────────────────────────────────
ENTITY_NAME    = "Customer"
KEY_COLUMN     = "CustomerId"
DISPLAY_COLUMN = "FullName"

# ── Build definition parts ───────────────────────────────────────────────────
parts = [
    {"path": "definition.json", "payload": to_base64({}), "payloadType": "InlineBase64"},
]

# Tracking maps
property_ids = {}
entity_type_id = generate_id()

# Build properties
properties = []
for col in columns:
    prop_id = generate_id()
    property_ids[col["name"]] = prop_id
    properties.append({
        "id": prop_id,
        "name": col["name"],
        "redefines": None,
        "baseTypeNamespaceType": None,
        "valueType": col["valueType"],
    })

key_prop_id     = property_ids[KEY_COLUMN]
display_prop_id = property_ids[DISPLAY_COLUMN]

# Entity type definition
entity_def = {
    "id": entity_type_id,
    "namespace": "usertypes",
    "baseEntityTypeId": None,
    "name": ENTITY_NAME,
    "entityIdParts": [key_prop_id],
    "displayNamePropertyId": display_prop_id,
    "namespaceType": "Custom",
    "visibility": "Visible",
    "properties": properties,
    "timeseriesProperties": [],
}
parts.append({
    "path": f"EntityTypes/{entity_type_id}/definition.json",
    "payload": to_base64(entity_def),
    "payloadType": "InlineBase64",
})

# Data binding (NonTimeSeries -> Lakehouse table)
binding_id = str(uuid.uuid4())
data_binding = {
    "id": binding_id,
    "dataBindingConfiguration": {
        "dataBindingType": "NonTimeSeries",
        "propertyBindings": [
            {"sourceColumnName": col["name"], "targetPropertyId": property_ids[col["name"]]}
            for col in columns
        ],
        "sourceTableProperties": {
            "sourceType": "LakehouseTable",
            "workspaceId": WORKSPACE_ID,
            "itemId": LAKEHOUSE_ID,
            "sourceTableName": TABLE_NAME,
        },
    },
}
parts.append({
    "path": f"EntityTypes/{entity_type_id}/DataBindings/{binding_id}.json",
    "payload": to_base64(data_binding),
    "payloadType": "InlineBase64",
})

print(f"Entity: {ENTITY_NAME} ({len(properties)} properties, key={KEY_COLUMN})")
print(f"Data binding: {TABLE_NAME} -> LakehouseTable")
print(f"Total definition parts: {len(parts)}")

---
## Step 3 — Create Ontology via REST API

Delete any existing ontology with the same name (for idempotency),
then create the new one with our full definition.

In [ ]:
# ── Delete existing ontology if present ──────────────────────────────────────
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/ontologies")
resp.raise_for_status()
existing = next(
    (o for o in resp.json().get("value", []) if o["displayName"] == ONTOLOGY_NAME),
    None,
)

if existing:
    print(f"Deleting existing ontology '{ONTOLOGY_NAME}' (id={existing['id']})...")
    del_resp = client.delete(f"v1/workspaces/{WORKSPACE_ID}/ontologies/{existing['id']}")
    del_resp.raise_for_status()
    print(f"   Waiting 30 seconds for cleanup...")
    time.sleep(30)
    print(f"   Cleanup complete.")

# ── Create ontology with full definition ─────────────────────────────────────
payload = {
    "displayName": ONTOLOGY_NAME,
    "description": "Customer360 ontology — Customer entity bound to Lakehouse Delta table.",
    "definition": {"parts": parts},
}

print(f"Creating ontology '{ONTOLOGY_NAME}'...")
resp = client.post(f"v1/workspaces/{WORKSPACE_ID}/ontologies", json=payload)

if resp.status_code == 201:
    result = resp.json()
    ONTOLOGY_ID = result["id"]
    print(f"Ontology created!  ID: {ONTOLOGY_ID}")

elif resp.status_code == 202:
    operation_id = resp.headers.get("x-ms-operation-id")
    retry_after  = int(resp.headers.get("Retry-After", "30"))
    print(f"   Operation {operation_id} — polling every {retry_after}s...")

    while True:
        time.sleep(retry_after)
        poll = client.get(f"v1/operations/{operation_id}")
        poll.raise_for_status()
        status = poll.json().get("status")
        print(f"   Status: {status}")

        if status == "Succeeded":
            # Fetch ontology ID
            list_resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/ontologies")
            list_resp.raise_for_status()
            ont = next(
                (o for o in list_resp.json().get("value", []) if o["displayName"] == ONTOLOGY_NAME),
                None,
            )
            ONTOLOGY_ID = ont["id"] if ont else "UNKNOWN"
            print(f"Ontology created!  ID: {ONTOLOGY_ID}")
            break
        elif status in ("Failed", "Cancelled"):
            print(f"Ontology creation {status.lower()}.")
            print(f"   {poll.json()}")
            ONTOLOGY_ID = None
            break
else:
    print(f"HTTP {resp.status_code}: {resp.text}")
    ONTOLOGY_ID = None
    resp.raise_for_status()

---
## Verification

In [ ]:
# List all ontologies in workspace
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/ontologies")
resp.raise_for_status()
ontologies = resp.json().get("value", [])

print(f"Ontologies in workspace ({len(ontologies)} total):")
for o in ontologies:
    marker = " <-- THIS ONE" if o.get("id") == ONTOLOGY_ID else ""
    print(f"   {o['displayName']} (ID: {o['id']}){marker}")

In [ ]:
# Store IDs for downstream notebooks
print("\n" + "=" * 60)
print("OUTPUTS — Copy these to the next notebook if needed")
print("=" * 60)
print(f"WORKSPACE_ID      = \"{WORKSPACE_ID}\"")
print(f"WORKSPACE_NAME    = \"{WORKSPACE_NAME}\"")
print(f"LAKEHOUSE_ID      = \"{LAKEHOUSE_ID}\"")
print(f"LAKEHOUSE_NAME    = \"{LAKEHOUSE_NAME}\"")
print(f"TABLE_NAME        = \"{TABLE_NAME}\"")
print(f"SEMANTIC_MODEL_ID = \"{SEMANTIC_MODEL_ID}\"")
print(f"ONTOLOGY_ID       = \"{ONTOLOGY_ID}\"")